In [ ]:
# Print current md-python tag
import importlib.metadata
import json
import subprocess

direct_url_files = [p for p in importlib.metadata.files('md-python') if 'direct_url.json' in str(p)]
if direct_url_files:
    direct_url = json.loads(direct_url_files[0].read_text())
    url = direct_url.get('url', '')
    if 'tags/' in url:
        tag = url.rstrip('.tar.gz').split('/')[-1]
    elif url.startswith('file://'):
        repo_path = url[len('file://'): ]
        result = subprocess.run(['git', 'describe', '--tags'], cwd=repo_path, capture_output=True, text=True)
        tag = result.stdout.strip() or '(no tag)'
    else:
        tag = '(unknown)'
    print(tag)
else:
    print('(no install metadata found)')

In [ ]:
# Setup: load env, create client
import os
from dotenv import load_dotenv
from md_python import MDClient

def create_md_client_from_env() -> MDClient:
    load_dotenv()
    api_token = os.getenv('MD_AUTH_TOKEN')
    assert api_token, 'MD_AUTH_TOKEN not set (check your .env)'
    base_url = os.getenv('MD_API_BASE_URL')
    return MDClient(api_token=api_token, base_url=base_url)

client = create_md_client_from_env()
print('Client initialized.')

In [ ]:
# Set the upload ID from a previous upload
upload_id = "76f2b5ad-2cc1-4a1b-aaf5-c71a40534343"  # <-- set this to your upload ID

In [ ]:
# Load experiment design and sample metadata from CSV (same as upload notebook)
import csv
from pathlib import Path
from typing import List, Tuple
from md_python import ExperimentDesign, SampleMetadata

LOCAL_DATA_DIR = Path('/Users/giuseppeinfusini/wd/md-repos/md-python/development/GI/do_not_save/test_data').resolve()
DESIGN_CSV_NAME = 'experiment_design_COMBINED.csv'

def _header_lower(header):
    return [h.strip().lower() if isinstance(h, str) else '' for h in header]

def _find_col(hl, names):
    for n in names:
        if n in hl:
            return hl.index(n)
    raise ValueError(f'CSV must have one of {names}')

def load_design_and_metadata(csv_path: Path) -> Tuple[ExperimentDesign, SampleMetadata]:
    with open(csv_path, 'r', encoding='utf-8') as f:
        raw = list(csv.reader(f))
    header = [h.strip() for h in raw[0]]
    hl = _header_lower(header)
    idx_fn     = _find_col(hl, ('filename', 'file'))
    idx_sample = _find_col(hl, ('sample_name', 'sample'))
    idx_cond   = _find_col(hl, ('condition', 'group'))
    data_rows  = [r for r in raw[1:] if r]

    design_data = [['filename', 'sample_name', 'condition']]
    for r in data_rows:
        design_data.append([
            r[idx_fn].strip()     if len(r) > idx_fn     else '',
            r[idx_sample].strip() if len(r) > idx_sample else '',
            r[idx_cond].strip()   if len(r) > idx_cond   else '',
        ])

    sample_col_indices = [i for i, h in enumerate(hl) if h not in ('filename', 'file')]
    sample_headers = [header[i] for i in sample_col_indices]
    seen, sample_rows = set(), []
    for r in data_rows:
        sn = r[idx_sample].strip() if len(r) > idx_sample else ''
        if sn and sn not in seen:
            seen.add(sn)
            sample_rows.append([r[i].strip() if len(r) > i else '' for i in sample_col_indices])

    return ExperimentDesign(data=design_data), SampleMetadata(data=[sample_headers] + sample_rows)

design_csv_path = LOCAL_DATA_DIR / DESIGN_CSV_NAME
assert design_csv_path.exists(), f'Missing {DESIGN_CSV_NAME} in {LOCAL_DATA_DIR}'
exp_design, sample_md = load_design_and_metadata(design_csv_path)
print(f'Design rows: {len(exp_design.data) - 1}, Sample rows: {len(sample_md.data) - 1}')

In [ ]:
# Find the initial intensity dataset for this upload
assert upload_id, 'Set upload_id in the cell above'
dataset = client.datasets.find_initial_dataset(upload_id)
assert dataset is not None, f'No initial dataset found for upload {upload_id}'
print('Initial dataset ID:', dataset.id)

In [ ]:
# Group samples by (cellline, family, drug) from sample metadata
from collections import defaultdict
from typing import Dict, Any

cols = sample_md.to_columns()
sample_names = cols['sample_name']
dose_values  = cols.get('dose', [])
celllines    = cols.get('cellline', [])
families     = cols.get('family', [])
drugs        = cols.get('drug', [])

# Build a dict: (cellline, family, drug) -> list of (sample_name, dose)
groups: Dict[tuple, List[Dict[str, Any]]] = defaultdict(list)
for i, sn in enumerate(sample_names):
    key = (
        celllines[i] if i < len(celllines) else '',
        families[i]  if i < len(families)  else '',
        drugs[i]     if i < len(drugs)     else '',
    )
    groups[key].append({'sample_name': sn, 'dose': dose_values[i] if i < len(dose_values) else '0'})

print(f'Found {len(groups)} group(s):')
for (cl, fam, drug), samples in sorted(groups.items()):
    print(f'  cellline={cl}, family={fam}, drug={drug} → {len(samples)} samples')

In [ ]:
# Create and submit a DoseResponseDataset for each (cellline, family, drug) group
from md_python import DoseResponseDataset

drc_jobs = {}  # key -> dataset_id

for (cl, fam, drug), samples in sorted(groups.items()):
    group_sample_names = [s['sample_name'] for s in samples]
    control_samples    = [s['sample_name'] for s in samples if str(s['dose']).strip() == '0']
    if not control_samples:
        control_samples = [group_sample_names[0]]
        print(f'  WARNING: no zero-dose controls for {drug}, using first sample as control')

    # Subset sample metadata to this group
    all_cols = sample_md.to_columns()
    header = list(all_cols.keys())
    indices = [i for i, sn in enumerate(all_cols['sample_name']) if sn in group_sample_names]
    group_rows = [[all_cols[h][i] for h in header] for i in indices]
    group_sample_md = SampleMetadata(data=[header] + group_rows)

    dataset_name = f'DRC_{drug}_{cl}_{fam}'
    builder = DoseResponseDataset(
        input_dataset_ids=[str(dataset.id)],
        dataset_name=dataset_name,
        sample_names=group_sample_names,
        control_samples=control_samples,
        sample_metadata=group_sample_md,
        dose_column='dose',
        log_intensities=True,
        use_imputed_intensities=True,
        normalise='none',
        span_rollmean_k=1,
        prop_required_in_protein=0.5,
    )

    drc_id = builder.run(client)
    drc_jobs[(cl, fam, drug)] = drc_id
    print(f'  Submitted {dataset_name} → dataset_id={drc_id}')

print(f'\nSubmitted {len(drc_jobs)} dose response job(s).')

In [ ]:
# Poll until all dose response jobs reach a terminal state
results = {}
for (cl, fam, drug), drc_id in drc_jobs.items():
    print(f'Waiting for DRC_{drug}_{cl}_{fam} (id={drc_id})...')
    ds = client.datasets.wait_until_complete(
        upload_id=upload_id,
        dataset_id=str(drc_id),
        poll_s=10,
        timeout_s=3600,
    )
    results[(cl, fam, drug)] = ds
    print(f'  → state={ds.state}')

print('\nAll done.')
results